# Arrival modelling and SIMUL8 arrival schedule
1. Estimate day-of-week demand coefficients and in-hours (WorkDay 09:00–17:00) vs out-of-hours (OOH) split ratios from 3,360 HIS admission records.
2. Generate an arrival schedule for SIMUL8 (100 trials × 8 weeks). Arrivals per day and shift follow a Poisson distribution, and timestamps are uniform within the shift window.
3. Check that the generated arrivals reproduce the empirical parameters.

Requires `data/Data_merge_Final.xlsx` (coursework data, not included).

In [1]:
import numpy as np
import pandas as pd

his = pd.read_excel("data/Data_merge_Final.xlsx", sheet_name="HIS_Admissions")
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

daily = his.groupby(["Date", "Day"])["Admitted"].sum().reset_index()
mean_daily = daily["Admitted"].mean()
coef = (daily.groupby("Day")["Admitted"].mean() / mean_daily).reindex(DAYS)

shift_tot = his.groupby(["Day", "Shift"])["Admitted"].sum().unstack()
workday_ratio = (shift_tot["WorkDay"] / shift_tot.sum(axis=1)).reindex(DAYS)

params = pd.DataFrame({"Demand coefficient": coef.round(3),
                       "WorkDay ratio (09-17)": workday_ratio.round(3),
                       "OOH ratio": (1 - workday_ratio).round(3)})
print(f"Records: {len(his):,}   Mean admissions per day: {mean_daily:.2f}   "
      f"Mean inter-arrival: {1440/mean_daily:.1f} min\n")
params

Records: 3,360   Mean admissions per day: 36.07   Mean inter-arrival: 39.9 min



,Demand coefficient,WorkDay ratio (09-17),OOH ratio
Day,,,
Monday,1.167,0.828,0.172
Tuesday,1.182,0.835,0.165
Wednesday,1.186,0.839,0.161
Thursday,1.217,0.838,0.162
Friday,1.171,0.827,0.173
Saturday,0.689,0.729,0.271
Sunday,0.387,0.492,0.508


## Generate the arrival schedule
For each day: WorkDay arrivals ~ Poisson(mean × coefficient × WorkDay ratio), placed uniformly in 09:00–17:00. OOH arrivals ~ Poisson(mean × coefficient × OOH ratio), placed uniformly across 00:00–09:00 and 17:00–24:00 (900 minutes in total).

In [2]:
def generate_arrivals(n_trials=100, n_weeks=8, seed=42, start="2025-01-06"):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, periods=7 * n_weeks, freq="D")
    rows = []
    for trial in range(1, n_trials + 1):
        for d_idx, date in enumerate(dates):
            day = date.day_name()
            lam = mean_daily * coef[day]
            for shift, ratio in [("WorkDay", workday_ratio[day]), ("OOH", 1 - workday_ratio[day])]:
                n = rng.poisson(lam * ratio)
                if shift == "WorkDay":
                    minutes = rng.uniform(540, 1020, n)
                else:                                     # map 0-900 onto 00:00-09:00 and 17:00-24:00
                    u = rng.uniform(0, 900, n)
                    minutes = np.where(u < 540, u, u + 480)
                for m in minutes:
                    rows.append((trial, date.date(), d_idx // 7 + 1, day, shift, m,
                                 d_idx * 1440 + m))
    out = pd.DataFrame(rows, columns=["Trial", "Date", "Week_Number", "Day_Name", "Shift",
                                      "Arrival_Minute_Of_Day", "Arrival_Minute_From_Trial_Start"])
    out = out.sort_values(["Trial", "Arrival_Minute_From_Trial_Start"]).reset_index(drop=True)
    out["Arrival_Time"] = pd.to_datetime(out["Arrival_Minute_Of_Day"], unit="m").dt.strftime("%H:%M")
    out["Entity_ID"] = np.arange(1, len(out) + 1)
    return out

arrivals = generate_arrivals()
print(f"{len(arrivals):,} arrivals generated ({arrivals.Trial.nunique()} trials × 8 weeks)")
arrivals.head()

202,261 arrivals generated (100 trials × 8 weeks)


,Trial,Date,Week_Number,Day_Name,Shift,Arrival_Minute_Of_Day,Arrival_Minute_From_Trial_Start,Arrival_Time,Entity_ID
0,1,2025-01-06,1,Monday,OOH,6.626043,6.626043,00:06,1
1,1,2025-01-06,1,Monday,OOH,125.817298,125.817298,02:05,2
2,1,2025-01-06,1,Monday,OOH,413.024198,413.024198,06:53,3
3,1,2025-01-06,1,Monday,OOH,511.867076,511.867076,08:31,4
4,1,2025-01-06,1,Monday,WorkDay,561.025808,561.025808,09:21,5


## Validation: generated vs empirical parameters

In [3]:
gen_daily = arrivals.groupby(["Trial", "Date", "Day_Name"]).size().reset_index(name="n")
gen_coef = (gen_daily.groupby("Day_Name")["n"].mean() / gen_daily["n"].mean()).reindex(DAYS)
gen_wd = (arrivals.groupby(["Day_Name", "Shift"]).size().unstack()
          .pipe(lambda t: t["WorkDay"] / t.sum(axis=1))).reindex(DAYS)

check = pd.DataFrame({"Empirical coef": coef, "Generated coef": gen_coef,
                      "Empirical WorkDay ratio": workday_ratio, "Generated WorkDay ratio": gen_wd}).round(3)
print(f"Mean arrivals per day: empirical {mean_daily:.2f}, generated {gen_daily.n.mean():.2f}\n")
check

Mean arrivals per day: empirical 36.07, generated 36.12



,Empirical coef,Generated coef,Empirical WorkDay ratio,Generated WorkDay ratio
Monday,1.167,1.171,0.828,0.827
Tuesday,1.182,1.182,0.835,0.834
Wednesday,1.186,1.189,0.839,0.840
Thursday,1.217,1.207,0.838,0.838
Friday,1.171,1.173,0.827,0.826
Saturday,0.689,0.689,0.729,0.728
Sunday,0.387,0.389,0.492,0.490


## Export for SIMUL8

In [4]:
arrivals.to_csv("simul8_arrivals_100trials_8weeks.csv", index=False)
print("Saved simul8_arrivals_100trials_8weeks.csv")

Saved simul8_arrivals_100trials_8weeks.csv
